## Проект 1. Как повышение цен связано с изменением выручки?
### Гипотеза: новая цена увеличила выручку, но успешных платежей могло стать меньше.




## Данные и период анализа

Я проверяю, как изменились выручка и число успешных платежей после повышения цен 1 марта 2026 года. Моя гипотеза: выручка выросла, но сокращение числа платежей уменьшило эффект повышения цены.

Для анализа я взяла свой сегмент: владельцев подписок, у которых были оплаты с баланса с июля по сентябрь 2025 года. Выручку считала только по успешным платежам, сгруппированным по месяцам и тарифам. Среднюю цену рассчитывала как выручку, делённую на число успешных платежей.

Сравнила январь и февраль с мартом, маем и июнем 2026 года. Апрель исключила из сравнения выручки, поскольку за 14 и 15 апреля нет записей о платежах. Чтобы учесть разную длительность периодов, пересчитала выручку и количество платежей на 30 дней.

Отток анализировала за все месяцы с января по июнь, включая апрель. Для каждого месяца считала долю завершившихся подписок среди активных на начало месяца. Показатель за период рассчитала как сумму завершений, делённую на сумму активных подписок на начала месяцев. Отдельно посмотрела долю отмен с причиной `too_expensive` среди всех отмен.


In [ ]:
# Сегмент: владельцы подписок с оплатами с баланса в июле–сентябре 2025 года.
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text
from prime.config import get_engine

engine = get_engine()

def q(sql):
    with engine.connect() as conn:
        conn.execute(text("set time zone 'UTC'"))
        return pd.read_sql(text(sql), conn)

# Условие моего сегмента зафиксировано из выданного задания.
CLIENT_FILTER = "client_id in (\n    select s.owner_client_id\n    from prime.payments p\n    join prime.subscriptions s on s.subscription_id = p.subscription_id\n    where p.payment_method = 'balance'\n      and p.paid_at >= '2025-07-01 00:00+00'\n      and p.paid_at <  '2025-10-01 00:00+00'\n)"
SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id,
           started_at, ended_at, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status
    from prime.payments p
    join prime.subscriptions s on p.subscription_id = s.subscription_id
    where s.owner_client_id in ({SEGMENT})
      and p.paid_at >= '2026-01-01' and p.paid_at < '2026-07-01'
""")

tariffs = q("select tariff_id, tariff_name from prime.tariffs")
print("Подписок:", len(subs), "| Попыток оплаты:", len(pay))

In [ ]:
# 1. Подготовка дат
# Дни, за которые нет платежей: они важны для выбора месяцев сравнения.
pay['paid_at'] = pd.to_datetime(pay['paid_at'],utc = True)
subs['started_at'] = pd.to_datetime(subs['started_at'],utc = True)
subs['ended_at'] = pd.to_datetime(subs['ended_at'],utc = True)

all_days = pd.date_range('2026-01-01','2026-06-30', freq = 'D', tz = "UTC")
days_with_payments=pd.DatetimeIndex(pay['paid_at'].dt.normalize().unique())
missing_days = all_days.difference(days_with_payments)

print(missing_days.strftime('%d.%m.%Y').to_list())
print(pay['status'].value_counts().to_string())

In [ ]:
# Названия тарифов для таблиц и графиков.
tariff_names = tariffs[['tariff_id','tariff_name']].drop_duplicates()
tariff_names

In [ ]:
# Проверки зерна данных и соединения до расчёта метрик.
assert subs['subscription_id'].notna().all()
assert subs['subscription_id'].is_unique, 'Повторяются subscription_id'
assert pay['payment_id'].notna().all()
assert pay['payment_id'].is_unique, 'Повторяются payment_id'
assert pay['subscription_id'].isin(subs['subscription_id']).all()
assert len(tariff_names) == tariff_names['tariff_id'].nunique()
print('Ключи уникальны, платежи относятся к выбранным подпискам.')


In [ ]:
# 2. Месячная выручка
# Для выручки берём только успешные платежи.
# Для сравнения месяцев приводим число платежей и выручку к 30 дням.
success = pay[pay['status'] =='success'].merge(
    subs[['subscription_id','tariff_id']],
    on='subscription_id',how='left',validate='m:1'   
)
success['month']=success['paid_at'].dt.strftime('%Y-%m')
monthly = success.groupby(['month','tariff_id'],as_index=False).agg(
    payments=('payment_id','size'),
    revenue=('amount','sum'))
monthly = monthly.merge(tariff_names, on='tariff_id',validate='m:1')
monthly['days']=pd.to_datetime(monthly['month']).dt.days_in_month
monthly['payments_30']=monthly['payments']/monthly['days']*30
monthly['revenue_30']=monthly['revenue']/monthly['days']*30
monthly[['month','tariff_name','payments','revenue','payments_30','revenue_30']].round(1)

In [ ]:
# График 1: выручка по месяцам в пересчёте на 30 дней.
# Апрель выделен отдельно: за два дня апреля платежей в данных нет.
monthly_total = monthly.groupby('month')['revenue_30'].sum()
colors=['lightgray' if m =='2026-04' else 'steelblue' for m in monthly_total.index]
plt.figure(figsize=(8,4))
plt.bar(monthly_total.index,monthly_total.values, color = colors)
plt.title('Выручка от успешных платежей по месяцам')
plt.ylabel('Выручка за 30 дней,руб')
plt.xlabel('Месяц 2026 г')
plt.grid(axis='y',alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Сравнение до и после изменения цен
# До: январь–февраль. После: март, май и июнь.
# Апрель не включаем в сравнение платежей из-за пропущенных дней.
# n30 — количество успешных платежей за 30 дней.
# r30 — выручка за 30 дней.
# price — средняя сумма одного успешного платежа.

compare = monthly[monthly['month'] != '2026-04'].copy()
compare['period'] = compare['month'].apply(lambda m: 'до' if m < '2026-03' else 'после')

period = compare.groupby(['tariff_id', 'period'], as_index=False).agg(
    payments=('payments', 'sum'),
    revenue=('revenue', 'sum'),
    days=('days', 'sum')
)
period['n30'] = period['payments'] / period['days'] * 30
period['r30'] = period['revenue'] / period['days'] * 30
period['price'] = period['revenue'] / period['payments']
period[['tariff_id', 'period', 'n30', 'price', 'r30']].round(2)


In [ ]:
# 4. Разложение изменения выручки
# Выручка = цена * количество успешных платежей.
# Вклад цены = (новая цена - старая цена) * новое количество.
# Вклад количества = (новое количество - старое количество) * старая цена.

# Сначала меняем число платежей при старой цене, затем цену при новом числе платежей.
before = period[period['period'] == 'до'].set_index('tariff_id')
after = period[period['period'] == 'после'].set_index('tariff_id')

factors = pd.DataFrame(index=before.index)
factors['n_before'] = before['n30']
factors['n_after'] = after['n30']
factors['price_before'] = before['price']
factors['price_after'] = after['price']
factors['revenue_before'] = before['r30']
factors['revenue_after'] = after['r30']
factors['change'] = factors['revenue_after'] - factors['revenue_before']
factors['tariff_name'] = factors.index.map(
    tariff_names.set_index('tariff_id')['tariff_name']
)


factors['price_effect'] = (factors['price_after'] - factors['price_before']) * factors['n_after']

factors['volume_effect'] = (factors['n_after'] - factors['n_before']) * factors['price_before']

factors[['tariff_name', 'n_before', 'n_after', 'price_before', 'price_after', 'change']].round(2)


In [ ]:
# Вклады цены и количества по всем тарифам.

r_before = factors['revenue_before'].sum()
r_after = factors['revenue_after'].sum()
delta = r_after - r_before
print('Выручка до:', round(r_before, 2))
print('Выручка после:', round(r_after, 2))
print('Изменение:', round(delta, 2))

price_total = factors['price_effect'].sum()
volume_total = factors['volume_effect'].sum()
print('Вклад цены:', price_total)
print('Вклад количества:', volume_total)
# Проверка точности факторной декомпозиции по каждому тарифу.
assert ((factors['price_effect'] + factors['volume_effect'] - factors['change']).abs() < 1e-6).all()
assert abs(price_total + volume_total - delta) < 1e-6


In [ ]:
# График 2: сравнение числа успешных платежей до и после по тарифам.

counts_plot = factors.set_index('tariff_name')[['n_before', 'n_after']]
counts_plot.columns = ['До', 'После']
counts_plot.plot(kind='bar', figsize=(7, 4), color=['gray', 'steelblue'], rot=0)
plt.title('Количество успешных платежей до и после')
plt.xlabel('Тариф')
plt.ylabel('Платежей за 30 дней')
plt.legend(title='Период')
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# График 3: влияние цены и числа платежей на изменение выручки.
# Отрицательный столбец означает снижение выручки из-за этого фактора.

effects_plot = factors.set_index('tariff_name')[['price_effect', 'volume_effect']]
effects_plot.columns = ['Цена', 'Число платежей']
effects_plot.plot(kind='bar', figsize=(8, 4), color=['steelblue', 'salmon'])
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Из чего сложилось изменение выручки')
plt.xlabel('Тариф')
plt.ylabel('Вклад в выручку за 30 дней, руб.')
plt.legend(title='Фактор')
plt.grid(axis='y', alpha=0.2)
plt.show()

In [ ]:
# 5. Отток подписок
# Отток в % = число закончившихся / число активных * 100.

# За апрель отток считаем: здесь используем даты окончания подписок, а не платежи.
churn_parts = []

for month_start in pd.date_range('2026-01-01', '2026-06-01', freq='MS', tz='UTC'):
    month_end = month_start + pd.DateOffset(months=1)
    active = subs[
        (subs['started_at'] <= month_start) &
        (subs['ended_at'].isna() | (subs['ended_at'] > month_start))
    ].copy()

    active['cancelled'] = (active['ended_at'].notna()) & (active['ended_at'] < month_end)

    part = active.groupby('tariff_id', as_index=False).agg(
        active_start=('subscription_id', 'size'),
        cancelled=('cancelled', 'sum')
    )
    part['month'] = month_start.strftime('%Y-%m')
    churn_parts.append(part)

churn = pd.concat(churn_parts, ignore_index=True)
churn = churn.merge(tariff_names, on='tariff_id')
churn['period'] = churn['month'].apply(lambda m: 'до' if m < '2026-03' else 'после')
churn['churn_pct'] = churn['cancelled'] / churn['active_start'] * 100
churn.head()


In [ ]:
# Отток до и после и долю отмен по причине too_expensive.

# Отток за оба периода по каждому тарифу
churn_period = churn.groupby(['tariff_id', 'period']).agg(
   active=('active_start', 'sum'),
    cancelled=('cancelled', 'sum')
 )
churn_period['churn_pct'] = (churn_period['cancelled']/churn_period['active'])*100
print(churn_period[['churn_pct']].round(3))

cancelled_subs = subs[
    (subs['ended_at'] >= '2026-01-01') &
    (subs['ended_at'] < '2026-07-01')
].copy()
cancelled_subs['period'] = cancelled_subs['ended_at'].dt.strftime('%Y-%m').apply(
    lambda m: 'до' if m < '2026-03' else 'после'
)
cancelled_subs['too_expensive'] = cancelled_subs['cancel_reason'] == 'too_expensive'

reasons = cancelled_subs.groupby(['tariff_id', 'period']).agg(
    cancellations=('subscription_id', 'size'),
    expensive=('too_expensive', 'sum')
)
# Доля too_expensive среди всех отмен в процентах
reasons['expensive_pct']= (reasons['expensive']/reasons['cancellations'])*100
reasons

In [ ]:
# График 4: отток по месяцам для каждого тарифа.
churn_plot = churn.pivot(index='month', columns='tariff_name', values='churn_pct')
churn_plot.plot(marker='o', figsize=(8, 4))
plt.title('Отток по месяцам для каждого тарифа')
plt.xlabel("Месяц")
plt.ylabel('Процент оттоков')
plt.legend(title='Тариф')
plt.tight_layout()
plt.show()

In [ ]:
# Итоговые показатели с единицами измерения.
print(f"Изменение выручки за 30 дней: {delta / r_before * 100:.3f}%")
print(f"Вклад цены: {price_total:,.2f} руб. за 30 дней")
print(f"Вклад количества платежей: {volume_total:,.2f} руб. за 30 дней")
display(churn_period[['churn_pct']].round(3))
display(reasons.round(3))


## Выводы и рекомендации

После повышения цен выручка в пересчёте на 30 дней выросла на 237 399,05 руб., или на 4,40%. Результаты по тарифам отличаются: у «Базового» выручка увеличилась на 250 788,35 руб., а у «Расширенного» снизилась на 13 389,30 руб. При этом число успешных платежей за 30 дней сократилось примерно на 11,9% и 11,5% соответственно.

Факторная декомпозиция показала, что повышение средней цены дало прирост выручки на 869 266,30 руб., а сокращение числа платежей уменьшило её на 631 867,26 руб. В сумме получился общий прирост на 237 399,05 руб. Таким образом, рост цены перекрыл снижение количества платежей по двум тарифам в целом.

Заметного роста оттока не наблюдается. У «Базового» показатель снизился с 3,835% до 3,697%, то есть на 0,138 процентного пункта. У «Расширенного» он почти не изменился: с 3,712% до 3,706%. Доля отмен по причине высокой цены у «Базового» осталась примерно на прежнем уровне, а у «Расширенного» снизилась с 26,49% до 23,60%.

Полученные результаты соответствуют моей гипотезе: выручка выросла, но число успешных платежей сократилось. Однако по сравнению двух периодов нельзя утверждать, что все изменения вызваны именно повышением цены. На результаты могли повлиять сезонность и изменение состава подписок.

Я бы пока сохранила текущие цены и продолжила следить за показателями отдельно по тарифам. Для «Расширенного» стоит подробнее изучить причины сокращения платежей, включая неуспешные списания и переходы на «Базовый». Для клиентов, отменяющих подписку из-за цены, можно протестировать предложение более доступного тарифа или временной скидки. Результат такого эксперимента нужно оценивать по выручке и повторным продлениям в сравнении с контрольной группой.
